# Músculo + Marcadores — Análisis EMG

Dataset: `data/musculoMasMarcadores.txt`

| Columna | Tipo | Descripción |
|---------|------|-------------|
| `Time` | float | Tiempo en segundos (paso 0.001 s → 1000 Hz) |
| `1 trap` | float | Canal EMG trapecio |
| `2 supraescapular` | float | Canal EMG supraescapular |
| `4 Memory` | bool | Marcador de contracción (1 = contracción activa) |

> **Objetivo inicial:** visualizar los 3 canales e identificar los instantes de contracción marcados por `4 Memory`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.stats import kurtosis, skew
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 4)

DATA_PATH = '../data/musculoMasMarcadores.txt'
SRATE = 1000  # Hz

## 1. Carga y descripción general

In [ ]:
df = pd.read_csv(DATA_PATH, sep='\t')

# Renombrar columnas para facilitar el acceso
df.columns = ['Time', 'trap', 'supra', 'memory']

# Rellenar NaN con interpolación lineal (huecos de adquisición)
df['trap']  = df['trap'].interpolate(method='linear')
df['supra'] = df['supra'].interpolate(method='linear')

t = df['Time'].values

print(f'Muestras   : {len(df):,}')
print(f'Duración   : {t[-1]:.1f} s  ({t[-1]/60:.1f} min)')
print(f'Frec.      : {SRATE} Hz')
print(f'Marcadores : {int(df["memory"].sum())} eventos (4 Memory = 1)')
print()
df[['trap', 'supra', 'memory']].describe().round(5)

## 2. Visualización completa — los 3 canales

Las líneas verticales rojas indican los instantes donde `4 Memory = 1` (contracción registrada).

In [ ]:
marker_times = t[df['memory'].values == 1]

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# ── Canal 1: Trapecio ────────────────────────────────────────────────────────
axes[0].plot(t, df['trap'].values, lw=0.4, color='steelblue')
for mt in marker_times:
    axes[0].axvline(mt, color='crimson', lw=0.8, alpha=0.7)
axes[0].set_title('1 trap  —  EMG Trapecio', fontsize=10)
axes[0].set_ylabel('Amplitud (mV)', fontsize=9)
axes[0].grid(True, alpha=0.25)

# ── Canal 2: Supraescapular ──────────────────────────────────────────────────
axes[1].plot(t, df['supra'].values, lw=0.4, color='darkorange')
for mt in marker_times:
    axes[1].axvline(mt, color='crimson', lw=0.8, alpha=0.7)
axes[1].set_title('2 supraescapular  —  EMG Supraescapular', fontsize=10)
axes[1].set_ylabel('Amplitud (mV)', fontsize=9)
axes[1].grid(True, alpha=0.25)

# ── Canal 3: Memory (marcador booleano) ──────────────────────────────────────
axes[2].fill_between(t, 0, df['memory'].values, color='crimson', alpha=0.6, step='mid')
axes[2].set_title('4 Memory  —  Marcador de contracción (booleano)', fontsize=10)
axes[2].set_ylabel('Activo', fontsize=9)
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(['0', '1'])
axes[2].grid(True, alpha=0.25)

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle(
    f'Vista completa  —  musculoMasMarcadores  |  {int(df["memory"].sum())} contracciones marcadas',
    fontsize=12
)
plt.tight_layout()
plt.show()

## 3. Zoom alrededor de los primeros marcadores

Ventana de ±2 s alrededor de cada marcador para ver la morfología de la contracción.

In [ ]:
WIN_PRE  = 2.0  # s antes del marcador
WIN_POST = 2.0  # s después del marcador
N_SHOW   = 6    # primeros N marcadores a visualizar

markers_to_show = marker_times[:N_SHOW]
fig, axes = plt.subplots(N_SHOW, 1, figsize=(14, N_SHOW * 2.5), sharex=False)

for i, mt in enumerate(markers_to_show):
    t_start = max(0, mt - WIN_PRE)
    t_end   = min(t[-1], mt + WIN_POST)
    mask    = (t >= t_start) & (t <= t_end)

    axes[i].plot(t[mask], df['trap'].values[mask],  lw=0.7, color='steelblue',  label='trap')
    axes[i].plot(t[mask], df['supra'].values[mask], lw=0.7, color='darkorange', label='supra', alpha=0.8)
    axes[i].axvline(mt, color='crimson', lw=1.2, ls='--', label=f'Marcador @ {mt:.3f} s')
    axes[i].set_title(f'Contracción #{i+1}  —  t = {mt:.3f} s', fontsize=9)
    axes[i].set_ylabel('mV', fontsize=8)
    axes[i].grid(True, alpha=0.25)
    if i == 0:
        axes[i].legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Tiempo (s)')
plt.suptitle(f'Zoom ±{WIN_PRE} s alrededor de los primeros {N_SHOW} marcadores', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Estadísticas por canal

In [ ]:
stats = []
for col, name in [('trap', '1 trap'), ('supra', '2 supraescapular')]:
    x = df[col].values
    stats.append({
        'Canal'    : name,
        'Media'    : np.mean(x),
        'Std'      : np.std(x),
        'RMS'      : np.sqrt(np.mean(x**2)),
        'Min'      : np.min(x),
        'Max'      : np.max(x),
        'Rango'    : np.max(x) - np.min(x),
        'Kurtosis' : kurtosis(x),
        'Skewness' : skew(x),
    })

stats_df = pd.DataFrame(stats).set_index('Canal')
stats_df.round(5)

## 5. Densidad espectral de potencia (PSD)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for col, name, color in [
    ('trap',  '1 trap',           'steelblue'),
    ('supra', '2 supraescapular', 'darkorange'),
]:
    freqs, psd = signal.welch(df[col].values, SRATE, nperseg=2048)
    ax.semilogy(freqs, psd, label=name, lw=1.2, color=color)

# Banda típica EMG
ax.axvspan(20, 150, alpha=0.10, color='green', label='Banda EMG (20–150 Hz)')
ax.set_xlim(0, 300)
ax.set_title('PSD — Canales EMG')
ax.set_xlabel('Frecuencia (Hz)')
ax.set_ylabel('PSD (mV²/Hz)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()